# Comparing radar and optical observations

The optical classifier was trained on Kaggle. Its training history and local, paired-sensor evaluation are versioned separately. Both models use a fixed 0.50 threshold for this comparison. Count a pixel only when its label, radar input and optical input are all valid. This population can differ from a single-sensor benchmark.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
out = root/'reports/m1-s2-kaggle-v2'
history = pd.read_csv(out/'history_s2.csv')
paired = json.loads((out/'paired_evaluation.json').read_text())['metrics']
fig, (curve, scores) = plt.subplots(1, 2, figsize=(11, 4))
curve.plot(history.epoch, history.val_iou_water, color='#1a6474')
curve.set(title='Optical training on Kaggle', xlabel='Epoch', ylabel='Validation water IoU', ylim=(0, 1))
curve.grid(axis='y', alpha=0.2)
x = np.arange(2)
for key, offset, color, label in [('radar_common', -0.17, '#38598a', 'Radar'), ('optical_common', 0.17, '#31816a', 'Optical')]:
    values = [paired[split][key]['iou_water'] for split in ['test', 'bolivia']]
    bars = scores.bar(x+offset, values, 0.34, color=color, label=label)
    scores.bar_label(bars, fmt='%.3f', padding=3)
scores.set(xticks=x, xticklabels=['Test', 'Bolivia'], ylim=(0, 1), ylabel='Water IoU', title='Common labelled pixels at threshold 0.50')
scores.legend(loc='lower right', frameon=False)
fig.tight_layout()
fig.savefig(out/'sensor_comparison.png', dpi=160, bbox_inches='tight')
plt.close(fig)

## Dynamic INT8 CPU compatibility

The deployment experiment first checks whether the CPU runtime can execute a dynamically quantized convolutional model. A synthetic input tests runtime compatibility only. Accuracy measurements require the labelled evaluation chips.

In [2]:
from hashlib import sha256
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

fp32 = out/'onnx_fp32/m1_s2_fp32.onnx'
config = json.loads((fp32.parent/'inference_config.json').read_text())
assert sha256(fp32.read_bytes()).hexdigest() == config['onnx_sha256']
quantized = out/'m1_s2_dynamic_int8.onnx'
quantize_dynamic(str(fp32), str(quantized), weight_type=QuantType.QInt8, per_channel=True)
record = {'source_sha256': config['onnx_sha256'], 'onnx_sha256': sha256(quantized.read_bytes()).hexdigest(),
          'bytes': quantized.stat().st_size, 'runtime': ort.__version__, 'provider': 'CPUExecutionProvider'}
try:
    session = ort.InferenceSession(str(quantized), providers=[record['provider']])
    logits = session.run(None, {'optical': np.zeros((1, 4, 512, 512), dtype='float32')})[0]
    record['runnable'] = bool(logits.shape == (1, 1, 512, 512) and np.isfinite(logits).all())
except ort.capi.onnxruntime_pybind11_state.NotImplemented as exc:
    record.update(runnable=False, error=str(exc))
(out/'dynamic_int8.json').write_text(json.dumps(record, indent=2)+'\n')
print(record)

{'source_sha256': '56feb3125762164681a155d9d561cd548037d9974737f7367194e09aafae9bbf', 'onnx_sha256': '8158af24bd2a8f99f73da06cb413b77639bd125281ef39db3422d40179a4a883', 'bytes': 24576621, 'runtime': '1.23.2', 'provider': 'CPUExecutionProvider', 'runnable': False, 'error': "[ONNXRuntimeError] : 9 : NOT_IMPLEMENTED : Could not find an implementation for ConvInteger(10) node with name '/encoder/conv1/Conv_quant'"}
